# Numerical Simulation of Quantum Tunnelling and Double-Well Qubit Dynamics

**Contents**
1. Introduction and objectives
2. Numerical method
   - 2.1 Discretisation
   - 2.2 Crank–Nicolson
   - 2.3 Setup and helper functions
4. Rectangular-barrier simulation
5. Numerical validation of the barrier simulation
6. Double-well potential (extension)
7. Connection to qubit-like state transfer
8. Discussion and conclusion
9. References and GenAI statement
- Appendix:
  - A: Animation of the barrier collision
  - B: Wavenumber distribution of the Gaussian wave packet

## 1. Introduction and objectives
- Quantum tunnelling is when a quantum particle approaching a potential barrier with less energy than the barrier height can tunnel through it. This differs from the prediction of classical mechanics, where a particle with less energy than the potential barrier would just be reflected.

- This project uses numerical solutions of the time-dependent Schrödinger equation to investigate:
1. **Rectangular barrier**: propagate a Gaussian wave packet towards a one-dimensional (rectangular) potential barrier, estimate transmission and reflection probabilities, and compare them with analytical results.
2. **Double-well potential**: calculate low-energy eigenstates and investigate how an initial state localised in one well evolves and transfers probability to the other well ($V(x)=ax^4-bx^2$).

The calculations use natural units, $\hbar=m=1$. The Gaussian wave packet propagating towards the rectangular barrier has central wave number $k_0=1$, giving an approximate mean kinetic energy $E\approx k_0^2/2=0.5$, below the barrier height $V_0=1$. The double-well extension explores how the energy splitting between the two lowest eigenstates sets the characteristic timescale for tunnelling.

- Application: the left and right wells can be associated with the computational basis states $|0\rangle$ and $|1\rangle$, so the double well can be a model for understanding qubit dynamics.

**Libraries:**
- NumPy: spatial grid, wavefunction and numerical calculations.
- matplotlib: plot results (and the animation in the Appendix).
- SciPy: `solve_banded` solves the tridiagonal linear system required for Crank–Nicolson; `eigsh` calculates the double-well eigenstates; `diags` builds the sparse Hamiltonian; `trapezoid` performs the energy average in Section 4.2.

## 2. Numerical method
The same finite-difference and Crank–Nicolson approach is used for both potentials, with differences coming from the potential profile (rectangular vs double-well) and the spatial grid.

### 2.1 Spatial discretisation and Hamiltonian
- Following a similar pattern used in the heat-equation workshop, where I constructed a tridiagonal matrix, built a right-hand side from the old temperature values and used `solve_banded` to calculate the new interior values.
- Difference: the heat equation used a real-valued temperature $T$, while the Schrödinger equation uses a complex-valued wavefunction $\psi$.

#### The Schrödinger equation

$$
i\hbar\frac{\partial\psi}{\partial t}
=
-\frac{\hbar^2}{2m}\frac{\partial^2\psi}{\partial x^2}
+V(x)\psi
$$
With $\hbar=m=1$, this simplifies to:
$$
i\frac{\partial\psi}{\partial t}
=
-\frac12\frac{\partial^2\psi}{\partial x^2}
+V(x)\psi
$$

Using the central-difference approximation for the second spatial derivative:
$$
\frac{\partial^2\psi}{\partial x^2}
\approx
\frac{\psi_{j+1}-2\psi_j+\psi_{j-1}}{dx^2},
$$
(this second derivative uses values at neighbouring grid points, producing a tridiagonal matrix).

The discrete Hamiltonian becomes:
$$
(H\psi)_j=
-\frac{\psi_{j+1}-2\psi_j+\psi_{j-1}}{2dx^2}
+V_j\psi_j.
$$

### 2.2 Crank–Nicolson time integration

For the quantum tunnelling problem, solve the time-dependent Schrödinger equation (with $\hbar=1$):
$$
i\frac{\partial\psi}{\partial t}=H\psi
$$
Applying Crank–Nicolson gives
$$
\boxed{
\left(I+\frac{i\,dt}{2}H\right)\psi^{n+1}
=
\left(I-\frac{i\,dt}{2}H\right)\psi^n
}
$$
Where:
- $I$ is the identity matrix.
- $H$ is the discretised Hamiltonian, representing the total energy operator.
- $\psi^n$ is the wavefunction at the current time step.
- $\psi^{n+1}$ is the wavefunction at the next time step.

The spatial domain is finite, $x\in[-L,L]$, with fixed boundary values $\psi(-L,t)=\psi(L,t)=0$. At each time step, the tridiagonal system is solved for the interior grid values. The method is useful here because it is implicit and preserves the wavefunction norm to numerical precision for a time-independent Hermitian Hamiltonian.

### 2.3 Setup and helper functions
The next two cells import the libraries and define the helper functions used throughout the notebook, so that each simulation later on is a short function call rather than a repeated block of code. The functions are described in the first table below; the second table describes the variables they use. (Each later section also lists the variables it introduces, in the same format.)

Functions needed by only one section are defined in that section: `T_analytic`, `T_averaged` and `barrier_transmission` in Section 4.2, and `lowest_eigenstates` in Section 5.2.

| Function | Purpose |
|---|---|
| `make_grid(L, N)` | uniform grid on $[-L, L]$ and its spacing |
| `total_probability(psi, dx)` | $\int\lvert \psi\rvert ^2dx$, approximated by $\sum_j\lvert \psi_j\rvert ^2\Delta x$ |
| `gaussian_packet(x, dx, x0, sigma, k0)` | normalised Gaussian wave packet (Section 3.1) |
| `rectangular_barrier(x, V0, d, edge_fix)` | potential array for a rectangular barrier (Sections 3.1 and 4.2) |
| `reflection_transmission(psi, x, dx, d)` | reflection and transmission probabilities (Section 4.2) |
| `crank_nicolson(psi0, V, dx, dt, T_end, ...)` | the time-stepping scheme of Section 2.2, wrapped in one function |

**Variables (Section 2.3 Helper functions).**

| Name | Meaning |
|---|---|
| `L`, `N` | half length of the domain and number of grid points |
| `x`, `dx` | grid array and the distance between neighbouring points |
| `psi` | wavefunction array (inside `crank_nicolson`: the working copy that is updated at every step) |
| `psi0` | initial wavefunction (never overwritten) |
| `x0`, `sigma`, `k0` | wave packet centre, position standard deviation and mean wavenumber |
| `V`, `V0`, `d` | potential array, barrier height and barrier width |
| `edge_fix` | `False`: barrier is $\lvert x\rvert \le d/2$; `True`: barrier is $-d/2\le x\lt d/2$ (explained in Section 4.2) |
| `eps` | tiny tolerance guarding against floating-point rounding at the barrier edges |
| `density` | probability density $\lvert \psi\rvert ^2$ |
| `R`, `T` | reflection and transmission probabilities |
| `dt`, `T_end` | time step and total simulated time |
| `n_steps` | number of time steps, `T_end / dt` |
| `n_int` | number of interior grid points |
| `H_diag` | diagonal entries of the discretised Hamiltonian |
| `ab` | left-hand side of the Crank–Nicolson matrix, in the 3-row banded storage required by `solve_banded` |
| `Hpsi` | H applied to the current interior wavefunction |
| `rhs` | right-hand side of the Crank–Nicolson equation, $(I-\tfrac{i\,dt}{2}H)\psi^n$ |
| `observe`, `record_every` | function returning the numbers to record, and how many steps between records |
| `snapshot_every` | number of steps between saved copies of $\psi$ |
| `times`, `obs` | times at which `observe` was called, and the values it returned |
| `snapshot_times`, `snapshots` | times of the saved copies of $\psi$, and the copies themselves (2D array) |

Plotting objects (`fig`, `ax`, `axes` ect.) are matplotlib figure and axes objects.

In [ ]:
# Import the libraries used throughout the notebook and set the default plot style
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded # solves the tridiagonal Crank-Nicolson system
from scipy.sparse import diags # assembles the sparse (tridiagonal) Hamiltonian
from scipy.sparse.linalg import eigsh # lowest energy eigenstates of the Hamiltonian
from scipy.integrate import trapezoid # numerical integration (used for energy averaging)

# Since the matplotlib version on my computer is different to the computers on campus
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Default matplotlib settings for on-screen presentation (same defaults as the workshop notebooks)
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (6.5, 4.0),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "lines.linewidth": 1.6,
})

In [ ]:
# Functions useful in many/every section (natural units, hbar = m = 1)
# included here to avoid repetition.

# Uniform grid on [-L, L] with N points
# returns the grid x and the spacing dx
def make_grid(L, N):
    x = np.linspace(-L, L, N)
    return x, x[1] - x[0]
    
# Total probability: the sum of |psi|^2 * dx -> approximates the integral of |psi|^2 dx
def total_probability(psi, dx):
    return np.sum(np.abs(psi)**2) * dx

# Normalised Gaussian wave packet centred on x0 (with width sigma and mean wavenumber k0)
def gaussian_packet(x, dx, x0, sigma, k0):
    psi = np.exp(-(x - x0)**2 / (4 * sigma**2)) * np.exp(1j * k0 * x)
    psi = psi / np.sqrt(total_probability(psi, dx))# normalise so the total probability is 1
    psi[0] = 0.0  # enforce the boundary conditions
    psi[-1] = 0.0
    return psi

# "Potential array for a rectangular barrier of height V0 and width d (centred on x = 0)
def rectangular_barrier(x, V0, d, edge_fix=False):
    eps = 1e-9                                  # guards against floating-point rounding at the barrier edges
    V = np.zeros_like(x)
    # edge_fix=False: barrier is |x| <= d/2, which contains one grid point too many (width d + dx)
    # edge_fix=True : barrier is -d/2 <= x < d/2, which contains exactly d/dx grid points (width d)
    if edge_fix:
        V[(x >= -d / 2 - eps) & (x < d / 2 - eps)] = V0
    else:
        V[np.abs(x) <= d / 2 + eps] = V0
    return V

# Reflection R (integrate |psi|^2 left of -d/2) and transmission T (integrate |psi|^2 right of + d/2) 
def reflection_transmission(psi, x, dx, d):
    density = np.abs(psi)**2
    R = np.sum(density[x < -d / 2]) * dx
    T = np.sum(density[x > d / 2]) * dx
    return R, T

# Evolve psi0 for a time T_end with the Crank-Nicolson scheme of Section 2.2 (psi = 0 at both ends)
def crank_nicolson(psi0, V, dx, dt, T_end, observe=None, record_every=1, snapshot_every=None):
    psi = psi0.astype(complex).copy() # work on a copy so that psi0 is not overwritten
    n_steps = int(round(T_end / dt))
    n_int = len(V) - 2  # interior grid points (the 2 boundary points stay at 0)

    # Diagonal of the discretised Hamiltonian (interior points only)
    H_diag = 1.0 / dx**2 + V[1:-1]

    # Left-hand Crank-Nicolson matrix (I + i dt/2 H) in 3-row banded storage; built once as V is time-independent
    ab = np.zeros((3, n_int), dtype=complex)
    ab[0, 1:] = -1j * dt / (4 * dx**2)
    ab[1, :] = 1.0 + 0.5j * dt * H_diag
    ab[2, :-1] = -1j * dt / (4 * dx**2)

    times = [0.0]
    # observe(psi) -> tuple of numbers, recorded every `record_every` steps
    obs = [observe(psi)] if observe else []
    # copy of psi is saved every `snapshot_every` steps (if given)
    # inital state is always included
    snapshot_times, snapshots = [0.0], [psi.copy()]

    for n in range(1, n_steps + 1):
        Hpsi = H_diag * psi[1:-1] - (psi[2:] + psi[:-2]) / (2 * dx**2)  # H applied to the current wavefunction
        rhs = psi[1:-1] - 0.5j * dt * Hpsi    # right-hand side (I - i dt/2 H) psi^n
        psi[1:-1] = solve_banded((1, 1), ab, rhs) # solve for psi^(n+1); boundaries remain 0

        if observe and (n % record_every == 0 or n == n_steps):
            times.append(n * dt)
            obs.append(observe(psi))
        if snapshot_every and (n % snapshot_every == 0 or n == n_steps):
            snapshot_times.append(n * dt)
            snapshots.append(psi.copy())
    # Returns a dictionary with keys: times, obs, snapshot_times, snapshots, psi
    return {"times": np.array(times), "obs": np.array(obs),
            "snapshot_times": np.array(snapshot_times), "snapshots": np.array(snapshots), "psi": psi}

## 3. Rectangular-barrier simulation
### 3.1 Potential and initial wave packet
The initial wave packet is chosen to have mean kinetic energy below the barrier height (for quantum tunnelling). In natural units, its approximate mean kinetic energy is $E\approx\frac{\hbar^2 k_0^2}{2m}\approx\frac{k_0^2}{2}=0.5$.

The rectangular barrier has height $V_0=1$ and width $d=2$, so the mean energy is below the barrier. The packet is initially centred at $x_0=-30$ and moves towards the barrier. Spatial domain: $x\in[-100,100]$, grid points: $N\approx4000$.

*Note:* the barrier is first defined as all grid points with $|x|\le d/2$. On a grid this includes one grid point more than the intended width $d$. The effect of this is examined in Section 4.2.

**Variables (Section 3.1 Setting up the Rectangular Potential Barrier).**

| Name | Meaning |
|---|---|
| `L` | half length of the domain |
| `N` | number of grid points |
| `x` | grid array |
| `dx` | distance between neighbouring points |
| `dt` | time step |
| `T_end` | total simulated time |
| `n_steps` | number of time steps |
| `V0`, `d` | potential barrier height $V_0$ and width $d$ |
| `V` | potential array on the grid |

In [ ]:
# Create the spatial grid and the rectangular potential barrier
L = 100.0
N = 4001   # more grid points (finer) than those used in workshop since wavefunctions can have rapid spatial oscillations
x, dx = make_grid(L, N)   # dx is the distance between neighbouring points

# Time parameters
dt = 0.05
T_end = 80.0
n_steps = int(round(T_end / dt))

# Rectangular potential barrier: height V0 and width d
V0 = 1.0
d = 2.0
V = rectangular_barrier(x, V0, d, edge_fix=False) # barrier is |x| <= d/2 (revisited in Section 4.2)

print(f"Spatial spacing dx = {dx:.4f}")
print(f"Number of time steps = {n_steps}")

# Plot the potential
plt.figure()
plt.plot(x, V, "r--")
plt.xlim(-45, 15)
plt.xlabel("Position x")
plt.ylabel("Potential V(x)")
plt.title("Rectangular potential barrier")
plt.show()

The initial state is a Gaussian wave packet (`psi0` in code):
$$
\psi(x,0)=
C\exp\left[-\frac{(x-x_0)^2}{4\sigma^2}\right]
e^{ik_0x}.
$$
Where the constant $C$ normalises the wavefunction, ensuring the total probability is one (as the particle must exist somewhere).
$$
\int |\psi|^2\,dx
\approx
\sum_j|\psi_j|^2\Delta x
\approx1
$$

**Variables (Section 3.1 Initialising the Gaussian wave packet).**

| Name | Meaning |
|---|---|
| `x0` | packet centre $x_0$ |
| `sigma` | position standard deviation $\sigma$ |
| `k0` | mean wavenumber $k_0$ |
| `psi0` | initial wavefunction (normalised, zero at both boundaries) |

In [ ]:
# Initialise the Gaussian wave packet
x0 = -30.0
sigma = 3.0
k0 = 1.0

# Construct the normalised wave packet (using gaussian_packet defined in Section 2.3)
psi0 = gaussian_packet(x, dx, x0, sigma, k0)

# Check the initial probability
print(f"Initial total probability = {total_probability(psi0, dx):.8f}")

# Plot the initial probability density
plt.figure()
# (np.abs(psi0)**2 calculates the square magnitude)
plt.plot(x, np.abs(psi0)**2)
plt.xlabel("Position x")
plt.ylabel("Probability density")
plt.title("Initial Gaussian wave packet")
plt.xlim(-45, 10)
plt.show()

#### Combining the rectangular potential barrier and the Gaussian wave packet

In [ ]:
# Combining the barrier and wave packet on a single plot
# Plot the initial conditions together: wave packet (on left) and barrier (on right)
fig, ax1 = plt.subplots(figsize=(10, 4))

ax1.plot(x, np.abs(psi0)**2, label="Probability density")
ax1.set_xlabel("Position, x")
ax1.set_ylabel(r"$|\psi(x,0)|^2$")
ax1.set_xlim(-45, 15)

ax2 = ax1.twinx()
ax2.plot(x, V, "r--", label="Potential barrier")
ax2.set_ylabel("Potential energy, V(x)")

plt.title("Initial wave packet and rectangular barrier")
plt.show()

### 3.2 Time evolution of the wave packet
The initial state is propagated using the Crank–Nicolson time integration equation (derived in Section 2.2). The following code records snapshots of the wavefunction every 5 time steps and the total probability at every time step.

**Variables (Section 3.2 Time evolution).** (The quantities used inside the time-stepping loop, `n_int`, `H_diag`, `ab`, `Hpsi` and `rhs`, are described in Section 2.3.)

| Name | Meaning |
|---|---|
| `snapshot_every` | number of time steps between saved snapshots of $\psi$ |
| `res` | dictionary returned by `crank_nicolson` |
| `times` | time of every recorded step |
| `norm_history` | $\int\lvert \psi\rvert ^2 dx$ at every step |
| `snapshot_times` | time of each saved snapshot |
| `snapshots` | list of copies of $\psi$ saved at intervals (a 2D array, one snapshot per row) |
| `psi_final` | wavefunction at the final time $t=T_{end}$ |

In [ ]:
# Evolve the wave packet with Crank-Nicolson (solving wavefunction each time step)
# Save snapshots and the total probability at every step
snapshot_every = 5   # save a snapshot every 5 steps (every 0.25 time units)

# Similar to Task 1.2 in Workshop 4
res = crank_nicolson(psi0, V, dx, dt, T_end,
                     observe=lambda psi: (total_probability(psi, dx),),
                     record_every=1, snapshot_every=snapshot_every)

# Unpack the results from dictionary
times = res["times"]  # time of every step
norm_history = res["obs"][:, 0]# total probability at every step
snapshot_times = res["snapshot_times"]# time of each saved snapshot
snapshots = res["snapshots"]  # 2D array: one saved wavefunction per row
psi_final = res["psi"]# wavefunction at t = T_end

print(f"Simulation finished at t = {n_steps * dt:.2f}")
print(f"Initial norm = {norm_history[0]:.8f}")
print(f"Final norm = {norm_history[-1]:.8f}")
print(f"Maximum norm error = {np.max(np.abs(norm_history - 1)):.2e}")

### 3.3 Wave packet evolution
#### Plotting the quantum wavefunction as it approaches the barrier

**Variables (Section 3.3 Plots).**

| Name | Meaning |
|---|---|
| `t_target` | a time at which a snapshot is wanted |
| `i` | index of the saved snapshot closest to the wanted time |

In [ ]:
# Plot selected probability-density snapshots to see how the packet approaches the barrier
# and splits into reflected and transmitted components
fig, ax = plt.subplots(figsize=(8.5, 4.5))

for t_target in [0, 20, 30, 40, 50, 60, 80]:
    i = np.argmin(np.abs(snapshot_times - t_target)) # snapshot closest to the target time
    ax.plot(x, np.abs(snapshots[i])**2, label=f"t = {snapshot_times[i]:.0f}")

# Show the barrier location
ax.axvspan(-d/2, d/2, alpha=0.2, color="grey", label="Potential barrier")
ax.set_xlim(-60, 60)
ax.set_xlabel("Position x")
ax.set_ylabel(r"Probability density $|\psi(x,t)|^2$")
ax.set_title("Quantum wave packet approaching a barrier")
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

Observations:
- At start $t=0$, the packet is centred at $x=-30$.
- As $t$ increases, the packet moves towards the barrier.
- After interacting with the barrier, some probability density appears back to the left and some appears to the right. Hence the probability density separates into reflected and transmitted components. This is evidence of tunnelling since the wave packet's mean energy is below the barrier height, yet there is still a transmitted component of probability density.
- Note: The wave packet contains a range of energies.

#### To investigate more closely: Stacked snapshots of the probability density of the wave packet were shown before, during and after the collision
Like in Workshop 5 (20/10/26) exploring numerical solutions of hyperbolic PDEs: 1D wave propagation. Using similar code to that used in the workshop to show the propagation of the wave packet.

**Variables (for stacked snapshots plot).**

| Name | Meaning |
|---|---|
| `snap_targets` | list of times at which snapshots are shown |
| `ts` | the target time of the panel being drawn |
| `xlim` | range of $x$ shown on the plots |
| `window` | boolean array: grid points inside `xlim` |
| `rho` | probability density $\lvert \psi\rvert ^2$ of the chosen snapshot |
| `ymax` | upper $y$-limit (largest |psi|^2)|

In [ ]:
# Stacked snapshots of |psi|^2 before, during and after the collision
snap_targets = [0.0, 25.0, 30.0, 35.0, 45.0, 80.0]   # times to show on snapshot
xlim = (-60, 60)
window = (x >= xlim[0]) & (x <= xlim[1])

# Shared y-limit: the largest |psi|^2 over all the chosen snapshots
idx = [np.argmin(np.abs(snapshot_times - ts)) for ts in snap_targets]
ymax = 0.8 * max((np.abs(snapshots[i])**2)[window].max() for i in idx)

fig, axs = plt.subplots(len(snap_targets), 1, figsize=(10, 9), sharex=True, sharey=True)
for ax, i, ts in zip(axs, idx, snap_targets):
    rho = np.abs(snapshots[i])**2
    ax.fill_between(x, 0, rho, color="C0", alpha=0.35)
    ax.plot(x, rho, "C0-", lw=1.2)
    ax.fill_between(x, 0, (V / V0) * 0.55 * ymax, color="C1", alpha=0.35)
    ax.set_ylabel(rf"$t = {ts:g}$")
    ax.set_ylim(0, ymax)
    ax.axhline(0, color="k", lw=0.5)
    # total area under the curve (should be 1 in every panel)
    ax.text(0.98, 0.8, rf"$\int|\psi|^2dx = {total_probability(snapshots[i], dx):.4f}$",
            transform=ax.transAxes, ha="right")

axs[-1].set_xlim(*xlim)
axs[-1].set_xlabel("$x$")
plt.suptitle(r"Wave packet $|\psi|^2$ before, during and after the collision", y=1.0)
plt.tight_layout(); plt.show()

## 4. Numerical Validation
Numerical accuracy is evaluated by examining the conservation of the wavefunction normalisation, the consistency of reflection and transmission probabilities, agreement with analytical solutions, and convergence under spatial grid refinement.

To check the accuracy of the numerical method:
1. Check that probability is conserved
2. Compare the transmission and reflection probabilities with analytical results
3. Check convergence as the grid is refined

### 4.1 Conservation of probability
The total probability is the spatial integral of the probability density, $\int|\psi(x,t)|^2\,dx$. It should remain close to one throughout the evolution. The plotted quantity is this integral, not the probability density at an individual point. (This cell uses `times` and `norm_history` from Section 3.2.)

In [ ]:
# Conservation of probability check: plot the total probability throughout the simulation
plt.figure()
plt.plot(times, norm_history)

plt.xlabel("Time t")
plt.ylabel(r"Total probability $\int |\psi|^2 dx$")
plt.title("Probability conservation")
plt.ylim(0.999, 1.001)
plt.show()

Hence the total probability is confirmed to stay at 1 throughout the simulation (the horizontal line at $\int|\psi|^2dx=1$). The largest deviation from 1 over all 1600 steps is about $7\times10^{-13}$ (printed in Section 3.2).

### 4.2 Transmission, reflection and analytical comparison
The final-time probabilities are estimated by integrating the probability density to the left and right of the barrier. (similar style of comparssion as exercise 3.4 in workshop 2).

**Variables (Section 4.2, reflection and transmission).**

| Name | Meaning |
|---|---|
| `R` | reflection probability: $\int\lvert \psi\rvert ^2dx$ for $x\lt -d/2$ at the final time |
| `T` | transmission probability: $\int\lvert \psi\rvert ^2dx$ for $x\gt d/2$ at the final time |

In [ ]:
# Reflection and transmission probabilities at the final time calculated
# Estimated by integrating the probability density either side of the barrier
R, T = reflection_transmission(psi_final, x, dx, d)

print(f"Reflection probability R = {R:.6f}")
print(f"Transmission probability T = {T:.6f}")
print(f"R + T = {R + T:.6f}")

The simulation gives:
- Reflection probability: $R\approx0.920610$
- Transmission probability: $T\approx0.079386$

The non-zero transmission probability (7.94% of the total probability), despite the mean energy being lower than the barrier height, is consistent with quantum tunnelling.
- $R+T\approx0.999995$

The probabilities either side of the barrier should add up to approximately 1. The code excludes the region inside the potential barrier, hence why $R+T$ is slightly below 1.

#### Analytical comparison
The **analytical transmission coefficient** for a particle with a single, definite energy $E<V_0$:
$$
T_{\mathrm{analytic}}=
\left[
1+\frac{V_0^2\sinh^2(\kappa d)}
{4E(V_0-E)}
\right]^{-1},
\qquad
\kappa=\sqrt{2(V_0-E)}
$$
(in natural units $\hbar=m=1$). For $E>V_0$, $\sinh$ is replaced by $\sin$, $V_0-E$ by $E-V_0$ in the denominator, and $\kappa$ by $k_2=\sqrt{2(E-V_0)}$.

Using $E\approx0.5$, $V_0=1$ and $d=2$ gives a transmission coefficient of approximately $0.071$. This is similar to $T=0.0794$ from the numerical simulation, but the two do not agree exactly. There are two reasons:

1. **Energy spread.** The analytical expression is for a stationary state with a single, definite energy. The numerical simulation propagated a Gaussian wave packet, which contains a range of energies distributed around its mean. The expected transmission is therefore the average of $T_{\mathrm{analytic}}(E_k)$, with $E_k=k^2/2$, over the packet's Gaussian distribution of wavenumbers $w(k)$ (only $k>0$ is incident from the left). (See Appendix B for derivation).
2. **Barrier width on the grid.** The code line `V[np.abs(x) <= d/2]` includes one extra grid point, so the barrier is $2.05$ wide instead of $2.0$ (for $dx=0.05$). A wider barrier lets less through. The alternative definition $-d/2\le x<d/2$ ("edge-corrected") contains exactly $d/dx$ grid points, so the width is exactly $d$ (in this case 2.0).

The following code defines the analytical functions and repeat the barrier simulation for both barrier definitions.

**Variables (Section 4.2, analytical comparison).**

| Name | Meaning |
|---|---|
| `E` | energy (a single value or an array) |
| `kappa`, `k2` | decay constant ($E\lt V_0$) and wavenumber ($E\gt V_0$) inside the barrier |
| `lo`, `hi`, `eq` | boolean masks for the cases $E\lt V_0$, $E\gt V_0$ and $E=V_0$ |
| `sigma_k` | width of the wave packet in wavenumber, $1/(2\sigma)$ |
| `k` | grid of wavenumbers over which the average is taken |
| `w` | weight $w(k)$ of each wavenumber in the packet |
| `xg`, `dxg`, `Vg` | grid, spacing and potential used inside `barrier_transmission` |
| `psi_start`, `psi_end` | initial and final wavefunctions inside `barrier_transmission` |
| `T_final` | transmission probability at the end of the run inside `barrier_transmission` |
| `T_single` | analytical $T$ for the single energy $E=k_0^2/2=0.5$ |
| `T_avg` | analytical $T$ averaged over the packet's energy distribution |
| `k_grid`, `w_grid` | the `k` and `w` arrays returned by `T_averaged` (reused in the plot) |
| `T_num_old`, `width_old` | numerical $T$ and effective barrier width for the original barrier ($\lvert x\rvert \le d/2$) |
| `T_num_fix`, `width_fix` | numerical $T$ and effective barrier width for the edge-corrected barrier |

In [ ]:
# Analytical transmission coefficients for a rectangular barrier

# Plane-wave transmission through a rectangular barrier of height V0 and width d (hbar = m = 1)
def T_analytic(E, V0, d):
    E = np.atleast_1d(np.asarray(E, dtype=float))
    T = np.empty_like(E)
    lo, hi, eq = E < V0, E > V0, E == V0  # three cases: E < V0, E > V0, E = V0
    kappa = np.sqrt(2 * (V0 - E[lo])) # decay constant inside the barrier (E < V0)
    T[lo] = 1.0 / (1.0 + V0**2 * np.sinh(kappa * d)**2 / (4 * E[lo] * (V0 - E[lo])))
    k2 = np.sqrt(2 * (E[hi] - V0)) # wavenumber inside the barrier (E > V0)
    T[hi] = 1.0 / (1.0 + V0**2 * np.sin(k2 * d)**2 / (4 * E[hi] * (E[hi] - V0)))
    T[eq] = 1.0 / (1.0 + V0 * d**2 / 2.0)  # limiting value at E = V0
    return T

# "Average T(E) over the packet's momentum distribution 
def T_averaged(k0, sigma, V0, d):
    sigma_k = 1.0 / (2.0 * sigma) # width of the packet in wavenumber
    k = np.linspace(max(1e-6, k0 - 8 * sigma_k), k0 + 8 * sigma_k, 4001)
    w = np.exp(-(k - k0)**2 / (2 * sigma_k**2))# Gaussian weight of each wavenumber
    # returns averaged T,the wavenumber grid k and the weights w of the distribution
    return trapezoid(T_analytic(0.5 * k**2, V0, d) * w, k) / trapezoid(w, k), k, w

# Repeat the barrier simulation (section 3) for a chosen grid (N points) and time step dt.
def barrier_transmission(N, dt, edge_fix, L=100.0, V0=1.0, d=2.0, x0=-30.0, sigma=3.0, k0=1.0, T_end=80.0):
    xg, dxg = make_grid(L, N)
    Vg = rectangular_barrier(xg, V0, d, edge_fix)
    psi_start = gaussian_packet(xg, dxg, x0, sigma, k0)
    psi_end = crank_nicolson(psi_start, Vg, dxg, dt, T_end)["psi"]
    T_final = reflection_transmission(psi_end, xg, dxg, d)[1]  # transmission probability at t = T_end
    # Returns the final transmission probability T and the effective barrier width (grid points in the barrier * dx)
    return T_final, int(np.sum(Vg > 0)) * dxg

In [ ]:
# Compare: analytical single-energy T, packet-averaged T, and the numerical T for both barrier definitions
T_single = T_analytic(0.5 * k0**2, V0, d)[0] # single energy E = k0^2 / 2
T_avg, k_grid, w_grid = T_averaged(k0, sigma, V0, d)# averaged over the packet's energy spread
T_num_old, width_old = barrier_transmission(N=4001, dt=0.05, edge_fix=False) # original barrier |x| <= d/2
T_num_fix, width_fix = barrier_transmission(N=4001, dt=0.05, edge_fix=True) # corrected barrier

print(f"Analytic, single energy E = 0.5     : T = {T_single:.4f}")
print(f"Analytic, averaged over the packet  : T = {T_avg:.4f}")
print(f"Numerical, original barrier (dx=0.05): T = {T_num_old:.4f}   effective width = {width_old:.3f}")
print(f"Numerical, edge-corrected barrier    : T = {T_num_fix:.4f}   effective width = {width_fix:.3f}")

**Results:**

| | $T$ |
|---|---|
| Analytical, single energy $E=0.5$ | 0.0707 |
| Analytical, averaged over the wave packet's energies | 0.0861 |
| Numerical, original barrier (width 2.05) | 0.0794 |
| Numerical, edge-corrected barrier (width 2.00) | 0.0861 |

The packet-averaged analytical value and the edge-corrected simulation agree **0.0861**, which is the main result of this comparison.

#### Plot: the energy-averaging picture
The plot shows why $0.0707$ becomes $0.0861$.

**Variables (energy-averaging plot).**

| Name | Meaning |
|---|---|
| `E_plot` | array of energies at which the plane-wave $T(E)$ is plotted |
| `E_k` | energy $k^2/2$ of each wavenumber in `k_grid` |

In [ ]:
# Plot T(E), the packet's energy distribution and the three transmission values
fig, ax = plt.subplots(figsize=(7, 4))
E_plot = np.linspace(0.02, 1.3, 400)
E_k = 0.5 * k_grid**2
ax.plot(E_plot, T_analytic(E_plot, V0, d), "k-", label="T(E), plane waves")
ax.axhline(T_single, color="C4", ls=":", label=f"single energy E = 0.5: {T_single:.3f}")
ax.axhline(T_avg, color="C2", ls="--", label=f"averaged over packet: {T_avg:.3f}")
ax.plot(0.5 * k0**2, T_num_fix, "^", color="r", ms=8, label=f"simulation (edge-corrected): {T_num_fix:.3f}")
ax.set_xlim(0, 1.3); ax.set_ylim(0, 0.45)
ax.set_xlabel("Energy E"); ax.set_ylabel("Transmission T")
ax2 = ax.twinx()                                  # packet's energy distribution, drawn at the bottom
ax2.fill_between(E_k, w_grid, color="C0", alpha=0.25)
ax2.set_ylim(0, 3.5 * w_grid.max()); ax2.set_yticks([]); ax2.grid(False)
ax.set_title("T(E) is convex near E = 0.5, so the energy spread raises T")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

- The black curve rising from left to right shows how easily a particle tunnels at each energy: low energy means very little gets through, higher energy means much more.
- The light blue Gaussian shows the spread of energies in the wave packet, centred on $E=0.5$.
- The red triangle near $0.086$ is the edge-corrected simulation.

#### Observations
- The single-energy value is $T=0.071$, but averaging over the packet's energy spread gives $T=0.086$.
- The gap arises because $T(E)$ curves upwards, so the high-energy side of the packet transmits more than the low-energy side loses.
- The simulation with the edge-corrected barrier gives $T=0.0861$, in agreement with the averaged analytical value.
- The original barrier gives $0.079$ because its effective width is $2.05$ rather than $2$.

### 4.3 Convergence study
Change the spatial grid spacing $\Delta x$ and the time step $\Delta t$, and check whether the results approach a consistent answer as the grid is refined. The barrier simulation is run at four grid spacings ($\Delta x=0.10, \ 0.05, \ 0.025, \ 0.0125$) with the reference time step $\Delta t=0.05$, once with the original barrier ($|x|\le d/2$) and once with the edge-corrected barrier. The time step is then halved at $\Delta x=0.05$ to test its effect on its own.

**Variables (Section 4.3 Convergence study).**

| Name | Meaning |
|---|---|
| `dx_runs` | list of `(N, dx)` pairs defining the four grids (with $L=100$, $dx=200/(N-1)$) |
| `N_run`, `dx_run` | number of grid points and spacing of the current run |
| `rows` | one tuple per run: `(dx, T_old, w_old, T_fix, w_fix)` |
| `T_old`, `w_old` | numerical $T$ and effective barrier width, original barrier |
| `T_fix`, `w_fix` | numerical $T$ and effective barrier width, edge-corrected barrier |
| `T_dt` | $T$ at $dx=0.05$ with the time step halved ($dt=0.025$), edge-corrected barrier |

In [ ]:
# Convergence in the grid spacing: run both barrier definitions at four grid spacings
dx_runs = [(2001, 0.1), (4001, 0.05), (8001, 0.025), (16001, 0.0125)]   # (N, dx) pairs; L = 100 so dx = 200/(N-1)
rows = []
for N_run, dx_run in dx_runs:
    T_old, w_old = barrier_transmission(N_run, 0.05, edge_fix=False)
    T_fix, w_fix = barrier_transmission(N_run, 0.05, edge_fix=True)
    rows.append((dx_run, T_old, w_old, T_fix, w_fix))

print(f"{'dx':>7} | {'T (|x|<=d/2)':>13} {'eff. width':>11} | {'T (edge fix)':>13} {'eff. width':>11}")
for r in rows:
    print(f"{r[0]:7.4f} | {r[1]:13.5f} {r[2]:11.4f} | {r[3]:13.5f} {r[4]:11.4f}")
print(f"Packet-averaged analytic T = {T_avg:.5f}")

# Time-step check: halve dt at dx = 0.05 (corrected barrier)
T_dt, _ = barrier_transmission(4001, 0.025, edge_fix=True)
print(f"\nTime-step check at dx = 0.05 (edge fix): dt = 0.05 -> {T_num_fix:.5f}, dt = 0.025 -> {T_dt:.5f}")

#### Observations from the table
- **Original barrier:** as $dx$ gets smaller, the effective width changes from $2.1$ down towards $2.0$. The barrier gets thinner, so more gets through and $T$ rises: $0.0731,\ 0.0794,\ 0.0827,\ 0.0844$ (still moving, so it has not converged).
- **Edge-corrected barrier:** the width is always exactly $2.0$, and $T$ is already about $0.086$ on the coarsest grid ($0.0859$, then $0.0861$ for all finer grids), matching the analytical value $0.08611$.
- Halving the time step changes nothing ($T=0.08607$ for both $dt=0.05$ and $dt=0.025$), so the time step is not the limiting factor.

#### Plotting the convergence
The plots turn the table into a picture. The convergence order is the slope of the error against $dx$ on log–log axes.

**Variables (convergence plot).**

| Name | Meaning |
|---|---|
| `dxs` | array of the four grid spacings |
| `T_old_arr`, `T_fix_arr` | arrays of $T$ for the original and edge-corrected barrier |
| `err_old`, `err_fix` | absolute error $\lvert T-T_{avg}\rvert $ for each barrier |
| `slope_old`, `slope_fix` | fitted slope of $\log(\text{error})$ against $\log(dx)$: the convergence order |

In [ ]:
# Plot the convergence: T against dx, and the error against dx on a log-log scale
dxs = np.array([r[0] for r in rows])
T_old_arr = np.array([r[1] for r in rows]); T_fix_arr = np.array([r[3] for r in rows])
err_old = np.abs(T_old_arr - T_avg); err_fix = np.abs(T_fix_arr - T_avg)
slope_old = np.polyfit(np.log(dxs), np.log(err_old), 1)[0] # fitted convergence order = slope on log-log axes
slope_fix = np.polyfit(np.log(dxs), np.log(err_fix), 1)[0]
print(f"Fitted order: original barrier {slope_old:.2f}, edge-corrected {slope_fix:.2f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(dxs, T_old_arr, "rs-", label="original: |x| <= d/2")
ax1.plot(dxs, T_fix_arr, "s-", label="edge-corrected")
ax1.axhline(T_avg, color="k", ls="--", label="packet-averaged analytic")
ax1.set_xlabel("Grid spacing dx"); ax1.set_ylabel("Transmission T"); ax1.legend()
ax1.set_title("Transmission against grid spacing")
ax2.loglog(dxs, err_old, "rs-", label=f"original (slope {slope_old:.2f})")
ax2.loglog(dxs, err_fix, "s-", label=f"edge-corrected (slope {slope_fix:.2f})")
ax2.loglog(dxs, err_old[0] * dxs / dxs[0], "k:", lw=1, label="slope 1")
ax2.loglog(dxs, err_fix[0] * (dxs / dxs[0])**2, "k--", lw=1, label="slope 2")
ax2.set_xlabel("Grid spacing dx"); ax2.set_ylabel("|T − T_avg|"); ax2.legend()
ax2.set_title("Error against the packet-averaged analytic T")
plt.tight_layout(); plt.show()

#### Observations
The figures compare the original barrier with the edge-corrected barrier: how $T$ changes with grid spacing, and how fast the error shrinks. The printed fitted convergence order was $0.97$ for the original barrier and $1.89$ for the edge-corrected barrier.

**Left plot:**
- The red line (original barrier) is not flat: it climbs as $dx$ shrinks and decreases as $dx$ grows. The blue line (edge-corrected barrier) is flat, sitting on the dashed line of the analytical value.
- Meaning: the original code gives an answer that depends on the grid, which is an indication something is slightly wrong. The reason is that the barrier was accidentally a bit too wide, and how much too wide depended on the grid.

**Right plot** (error on a log–log scale):
- The red line (original barrier) falls along the dotted "slope 1" line. The fitted slope of $0.97$ means convergence is first order.
- The blue line (edge-corrected barrier) falls along the dashed "slope 2" line. The fitted slope of $1.89$ means convergence is close to second order.

**My original barrier was slightly too wide, and by how much depended on the grid. Fixing that made the simulation agree with the theory and converge at the speed it should.**

## 5. Double-Well Potential (extension)
### 5.1 Potential
- Using a symmetric double-well potential: $V(x)=ax^4-bx^2,\qquad a,b>0.$
- The two minima are at $x_{\min}=\pm\sqrt{\frac{b}{2a}}$
- A central barrier separates the wells
- Barrier height: $V(0)-V(x_{\min})=\frac{b^2}{4a}$

Aim: to calculate low-energy eigenstates and use their superposition to construct a state initially localised in one well.

The double well has its own, smaller spatial grid ($L=15$), so its variables carry the suffix `_dw` to keep them separate from the barrier simulation of Section 3.

#### Setting up the double-well potential

**Variables (Section 5.1 Double-well potential).**

| Name | Meaning |
|---|---|
| `L_dw`, `N_dw` | half length of the double-well domain and number of grid points |
| `x_dw`, `dx_dw` | grid array and grid spacing for double well |
| `a`, `b` | parameters controlling the shape of the potential |
| `V_dw` | potential array $V(x)=ax^4-bx^2$ on the grid |
| `x_min` | position of the right minimum (the left minimum is at `-x_min`) |
| `barrier_height` | height of the central barrier above the minima, $b^2/(4a)$ |

In [ ]:
# Set up the double-well potential on its own spatial grid (L=15)
L_dw = 15.0
N_dw = 1501
x_dw, dx_dw = make_grid(L_dw, N_dw)

# Double-well potential parameters (control shape)
a = 0.01
b = 0.2

# Construct the potential
V_dw = a * x_dw**4 - b * x_dw**2

# Theoretical well locations and barrier height
x_min = np.sqrt(b / (2 * a))# position of the 2 minima (at +x_min and -x_min)
barrier_height = b**2 / (4 * a)# height of the central barrier

print(f"Spatial spacing dx = {dx_dw:.4f}")
print(f"Left minimum: x = {-x_min:.3f}")
print(f"Right minimum: x = {x_min:.3f}")
print(f"Central barrier height = {barrier_height:.3f}")

plt.figure()
plt.plot(x_dw, V_dw)
plt.xlabel("Position x")
plt.ylabel("Potential V(x)")
plt.title("Double-well potential")
# Add limits for zoomed-in view of the double well
plt.xlim(-6, 6)
plt.ylim(-1.2, 0.5)
plt.show()

### 5.2 Energy eigenstates
Stationary states satisfy the time-independent Schrödinger equation: $\hat H\phi_n=E_n\phi_n$, with the energy eigenvalue $E_n$ and its corresponding eigenfunction $\phi_n$. $\hat H$ is the Hamiltonian operator.

The function `lowest_eigenstates` below builds the discretised Hamiltonian of Section 2.1 as a tridiagonal sparse matrix (`diags` assembles the diagonals into a sparse matrix, which is useful since most entries of the matrix are zero), finds its lowest eigenvalues with `eigsh`, sorts them from lowest to highest, adds the zero boundary values back in, and normalises each eigenstate so that
$\int_{-L}^{L}|\phi_n(x)|^2\,dx\approx1$.

The four lowest-energy states are calculated to examine the symmetry and nodal structure of the eigenfunctions. This allows an understanding of how a quantum particle behaves in the double-well potential, and how it can move between the left and right wells.

- $E_0$: ground state, the lowest possible energy.
- $E_1$: first excited state.
- $E_2$: second excited state.
- $E_3$: third excited state.

**Variables (Section 5.2 Energy eigenstates).**

| Name | Meaning |
|---|---|
| `k` | number of lowest eigenstates to calculate |
| `n_int` | number of interior grid points (the two boundary points are excluded, where $\psi=0$) |
| `H_diag` | main diagonal of the Hamiltonian matrix, $1/dx^2+V_j$ |
| `H_off` | the two neighbouring diagonals, $-1/(2dx^2)$ |
| `H` | the tridiagonal Hamiltonian as a sparse matrix |
| `E`, `vec` | eigenvalues and eigenvectors returned by `eigsh` |
| `order` | indices that sort the eigenvalues from lowest energy to highest |
| `energies` | the four lowest energy eigenvalues $E_0,\dots,E_3$ |
| `phi` | array of eigenstates across the full grid including the zero boundaries; column `n` is $\phi_n(x)$ |

In [ ]:
# Find the lowest 4 energy states:

# Calculate the 4 lowest energy eigenvalues and eigenstates of H = -1/2 d2/dx2 + V (psi = 0 at both ends)
def lowest_eigenstates(V, dx, k=4):
    
    n_int = len(V) - 2 # exclude the two boundary points, where psi = 0

    # Hamiltonian diagonal and off-diagonal elements
    H_diag = 1.0 / dx**2 + V[1:-1] # main diagonal
    H_off = -1.0 / (2.0 * dx**2) * np.ones(n_int - 1) # the two neighbouring diagonals

    # Assemble the tridiagonal Hamiltonian as a sparse matrix (most entries are zero)
    H = diags([H_off, H_diag, H_off], offsets=[-1, 0, 1], format="csr")

    # Calculate the k lowest eigenvalues and eigenvectors using eigsh, sorted from lowest energy to highest
    E, vec = eigsh(H, k=k, which="SA")
    order = np.argsort(E)
    E, vec = E[order], vec[:, order]

    # Add the zero boundary values back into each eigenstate
    phi = np.zeros((len(V), k), dtype=complex)
    phi[1:-1, :] = vec

    # Normalise each eigenstate so that the integral of |phi|^2 is 1
    for n in range(k):
        phi[:, n] /= np.sqrt(total_probability(phi[:, n], dx))
    return E, phi


energies, phi = lowest_eigenstates(V_dw, dx_dw, k=4)

print("Lowest four energy eigenvalues:")
for n in range(len(energies)):
    print(f"E_{n} = {energies[n]:.6f}")

Results:
- $E_0=-0.588647$
- $E_1=-0.576136$
- $E_2=0.011086$
- $E_3=0.225485$

The first two levels are close in energy compared with the gap to the higher states.

#### Plot the eigenstates: $\phi_n(x)$ and $|\phi_n(x)|^2$ against position $x$

In [ ]:
# Plot both the eigenfunctions and their probability densities
fig, axes = plt.subplots(4, 2, figsize=(12, 12))

for n in range(4):
    # Left column: eigenfunction (shows sign and symmetry)
    axes[n, 0].plot(x_dw, phi[:, n].real, lw=2)
    axes[n, 0].axhline(0, color="black", lw=0.8, linestyle="--")
    axes[n, 0].set_title(f"State {n}: Eigenfunction ($E={energies[n]:.4f}$)")
    axes[n, 0].set_ylabel(r"$\phi_n(x)$")

    # Right column: probability density
    axes[n, 1].plot(x_dw, np.abs(phi[:, n])**2, lw=2)
    axes[n, 1].set_title(f"State {n}: Probability density")
    axes[n, 1].set_ylabel(r"$|\phi_n(x)|^2$")

# Label the x-axis only on the bottom row
for ax in axes[-1, :]:
    ax.set_xlabel("Position x")

plt.tight_layout()
plt.show()

Observations from the eigenstate plots:
- The ground state $\phi_0(x)$ is symmetric.
- The first excited state $\phi_1(x)$ is antisymmetric and changes sign at the centre.
- The probability densities ($|\phi_n(x)|^2$) of both are symmetric (squaring the magnitude removes the sign).
- $E_0$ and $E_1$ have near degeneracy (close in energy)

This near degeneracy allows a superposition to form a wavefunction localised in one well. As the relative phase between the states evolves over time, the probability distribution shifts between the wells, illustrating quantum tunnelling. (This is illustrated in section 5.4).

### 5.3 Energy splitting and predicted transfer time
- The energy difference between the two lowest states is $\Delta E=E_1-E_0$ (this is the **energy splitting**).
- For an initial state formed from an equal superposition of these two states, the relative phase evolves at a rate set by $\Delta E/\hbar$.
- This determines the characteristic timescale of the tunnelling, with the approximate first transfer time $t_{\mathrm{transfer}}\approx\frac{\pi}{\Delta E}$ (when $\hbar=1$).
- The smaller the energy splitting, the longer the tunnelling timescale.

The prediction is calculated here and compared with the simulation in Section 5.5.

**Variables (Section 5.3 Energy splitting).**

| Name | Meaning |
|---|---|
| `delta_E` | energy splitting $\Delta E=E_1-E_0$ |
| `predicted_transfer_time` | predicted first transfer time $\pi/\Delta E$ |

In [ ]:
# Calculate the energy splitting and the predicted first transfer time
delta_E = energies[1] - energies[0]
predicted_transfer_time = np.pi / delta_E     # hbar = 1

print(f"Ground-state energy E0 = {energies[0]:.6f}")
print(f"First excited energy E1 = {energies[1]:.6f}")
print(f"Energy splitting Delta E = {delta_E:.6f}")
print(f"Predicted first transfer time = {predicted_transfer_time:.3f}")

**Results:**
- From the first two levels: $\Delta E=E_1-E_0=-0.576136-(-0.588647)=0.012512$
- Predicted first transfer time $=251.089$ (in natural time units)

### 5.4 Initial state localised in one well
For a symmetric double well, the two lowest-energy eigenstates are approximately symmetric and antisymmetric combinations of states localised in the left and right wells. Combining them with equal amplitudes and an appropriate relative sign produces an initial state localised predominantly in the left well. This state is then propagated with the same Crank–Nicolson method.

**Variables (Section 5.4 Initial state).**

| Name | Meaning |
|---|---|
| `phi0`, `phi1` | copies of the ground and first excited states |
| `left_half` | boolean array: grid points with $x\lt 0$ |
| `overlap_left` | overlap of `phi0` and `phi1` over the left half; its sign tells whether `phi1` must be flipped (eigenvector signs are arbitrary) |
| `psi_left` | initial state: equal superposition of `phi0` and `phi1`, localised in the left well |

In [ ]:
# Plotting a state localised in the left well.

# Use copies of the two lowest eigenstates
phi0 = phi[:, 0].copy()
phi1 = phi[:, 1].copy()

# Eigenvector signs are arbitrary.
# Choose the sign of phi1 so the combination is localised on the left.
left_half = x_dw < 0
overlap_left = np.sum(np.conjugate(phi0[left_half]) * phi1[left_half]) * dx_dw
if np.real(overlap_left) < 0:
    phi1 = -phi1

# Initial state: equal superposition of the two lowest energy states
psi_left = (phi0 + phi1) / np.sqrt(2)

# Normalise to account for numerical rounding
psi_left /= np.sqrt(total_probability(psi_left, dx_dw))

print(f"Initial total probability = {total_probability(psi_left, dx_dw):.8f}")

plt.figure()
plt.plot(x_dw, np.abs(psi_left)**2)
plt.axvline(0, color="grey", linestyle="--")
plt.xlabel("Position x")
plt.ylabel(r"Probability density $|\psi(x,0)|^2$")
plt.title("Initial state localised in the left well")
plt.show()

### 5.5 Time evolution and probability transfer between wells
The probability of finding the particle in each half of the domain is estimated as

$P_L(t)=\int_{-L}^{0}|\psi(x,t)|^2\,dx, \qquad P_R(t)=\int_{0}^{L}|\psi(x,t)|^2\,dx$

The simulation runs for $1.2$ times the predicted first transfer time, so that the maximum of $P_R$ lies inside the run rather than at its end. The observed transfer time is then an independent check of the prediction.

**Variables (Section 5.5 Time evolution).**

| Name | Meaning |
|---|---|
| `dt_dw` | time step |
| `T_end_dw` | total simulated time, $1.2\times$ `predicted_transfer_time` |
| `observe_wells` | function returning $P_L$, $P_R$ and the total probability for a wavefunction |
| `res_dw` | dictionary returned by `crank_nicolson` |
| `times_dw` | time of every recorded step |
| `P_left`, `P_right` | $P_L(t)$ and $P_R(t)$ at every recorded step |
| `norm_dw` | total probability at every recorded step |
| `snapshot_times_dw`, `snapshots_dw` | times of the saved snapshots (every 50 steps) and the snapshots themselves (2D array) |

In [ ]:
# Apply Crank-Nicolson to the double-well potential

# Time settings
dt_dw = 0.02
T_end_dw = 1.2 * predicted_transfer_time    # run past the predicted first transfer, so the peak lies inside the run

# Quantities recorded at every step: probability in the left half, right half and total prob.
observe_wells = lambda psi: (np.sum(np.abs(psi[x_dw < 0])**2) * dx_dw,
                             np.sum(np.abs(psi[x_dw > 0])**2) * dx_dw,
                             total_probability(psi, dx_dw))

res_dw = crank_nicolson(psi_left, V_dw, dx_dw, dt_dw, T_end_dw,
                        observe=observe_wells, record_every=1, snapshot_every=50)

# Unpack the results from res_dw dictionary
times_dw = res_dw["times"]  # time of every recorded step
P_left = res_dw["obs"][:, 0]  # probability in the left half, P_L(t)
P_right = res_dw["obs"][:, 1]  # probability in the right half, P_R(t)
norm_dw = res_dw["obs"][:, 2]  # total probability
snapshot_times_dw = res_dw["snapshot_times"] # time of each saved snapshot
snapshots_dw = res_dw["snapshots"] # 2D array: one saved wavefunction per row

print(f"Simulation finished at t = {times_dw[-1]:.2f}")
print(f"Final total probability = {norm_dw[-1]:.8f}")
print(f"Maximum recorded norm error = {np.max(np.abs(norm_dw - 1)):.2e}")

In [ ]:
# Probability in each well over time
plt.figure(figsize=(8, 5))
plt.plot(times_dw, P_left, label="Left well")
plt.plot(times_dw, P_right, "r", label="Right well")
plt.xlabel("Time t")
plt.ylabel("Probability")
plt.title("Quantum tunnelling between the two wells")
plt.legend()
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

#### Comparing the observed transfer time with the prediction

**Variables (comparison).**

| Name | Meaning |
|---|---|
| `max_index` | index of the largest value of $P_R$ |
| `max_P_right` | maximum right-well probability |
| `observed_transfer_time` | time at which $P_R$ is largest |

In [ ]:
# Compare the observed transfer time (peak of P_R) with the prediction from the energy splitting

# Calculate the maximum right-well probability (and when it occurs)
max_index = np.argmax(P_right) # index of the largest right-well probability
max_P_right = P_right[max_index]
observed_transfer_time = times_dw[max_index]

print(f"Maximum right-well probability: {max_P_right:.4f}")
print(f"Observed transfer time: {observed_transfer_time:.2f}")
print(f"Predicted transfer time: {predicted_transfer_time:.2f}")
print(f"Difference: {abs(observed_transfer_time - predicted_transfer_time):.2f}")

Observation:
- Initially, the state is localised in the left well, so $P_L(0)\approx1$ and $P_R(0)\approx0$.
- $P_R$ then rises to a maximum and starts to fall again as the probability transfers back, so probability oscillates between the wells.
- The maximum right-well probability is approximately $0.997$, indicating that transfer is nearly complete.
- The observed time of this maximum is approximately $251.10$, in close agreement with the energy-splitting estimate of $251.09$ natural time units (a difference of $0.01$, smaller than the time step $dt=0.02$). This supports the expected relationship between the splitting and the tunnelling timescale.

#### Plot the tunnelling dynamics

**Variables (snapshot plot).**

| Name | Meaning |
|---|---|
| `snap_targets_dw` | times at which snapshots are shown: about a quarter, a half and all of the first transfer |

In [ ]:
# Plot snapshots of the changing probability density
snap_targets_dw = [0, 63, 126, 251]   # times to show (about 1/4, 1/2 and all of the first transfer)

fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, t_target in zip(axes.flat, snap_targets_dw):
    i = np.argmin(np.abs(snapshot_times_dw - t_target)) # snapshot closest to the target time
    ax.plot(x_dw, np.abs(snapshots_dw[i])**2)
    ax.axvline(0, color="grey", linestyle="--")
    ax.set_title(f"t = {snapshot_times_dw[i]:.2f}")
    ax.set_xlabel("Position x")
    ax.set_ylabel(r"$|\psi(x,t)|^2$")

plt.tight_layout()
plt.show()

Observation:

- $t=0$: initially, probability is concentrated in the left well.
- $t\approx63$: probability is beginning to shift and redistribute between the wells.
- $t\approx126$: intermediate stage, with substantial probability in both wells.
- $t\approx251$: probability is almost entirely in the right well (the first transfer maximum).

This is evidence of oscillation of probability between the wells.

The combination of these snapshots of the changing probability density over position and the plot of probability in each well over time illustrates the **tunnelling dynamics** and shows the transfer timescale (which agrees with the estimate from the energy splitting).

#### Check probability conservation

In [ ]:
# Check probability conservation in the double-well simulation
plt.figure()
plt.plot(times_dw, norm_dw)
plt.xlabel("Time t")
plt.ylabel(r"Total probability $\int |\psi|^2 dx$")
plt.title("Probability conservation")
plt.ylim(0.99, 1.01)
plt.tight_layout()
plt.show()

Observation:
The total probability remains close to 1 (therefore conserved) throughout the double-well simulation (maximum deviation about $10^{-12}$), supporting numerical stability and normalisation of the time evolution.

## 6. Connection to qubit-like state transfer
### 6.1 The qubit

The left- and right-localised states of the symmetric double-well potential can be used as approximate representations of the computational basis states of a qubit:
- $|0\rangle$: a particle localised in the left well.
- $|1\rangle$: a particle localised in the right well.

(These localised states are approximate rather than exact energy eigenstates.)
A general superposition of the two states can be written as

$$
|\psi\rangle=\alpha|0\rangle+\beta|1\rangle,
\qquad |\alpha|^2+|\beta|^2=1,
$$

where $\alpha$ and $\beta$ are complex probability amplitudes. The probabilities of finding the particle in the left and right wells are given by $|\alpha|^2$ and $|\beta|^2$, respectively.

For a symmetric double well, the two lowest energy eigenstates can be approximated by equal-amplitude superpositions of the localised states:

$$
|E_0\rangle\approx
\frac{|0\rangle+|1\rangle}{\sqrt{2}},
\qquad
|E_1\rangle\approx
\frac{|0\rangle-|1\rangle}{\sqrt{2}}.
$$

These symmetric and antisymmetric energy eigenstates have equal probabilities of being found in either well. Their different relative phases lead to different interference behaviour, which underlies tunnelling between the wells.

The double-well simulation therefore provides a model of qubit-like state transfer through quantum tunnelling. An initially left-localised state can evolve towards the right well and subsequently return, demonstrating the exchange of probability between the two localised states. The characteristic transfer timescale is related to the energy difference between the two lowest energy eigenstates.

### 6.2 How good is the two-state picture?
This checks how well the double well behaves like a "two-state" (qubit-like) system.

A particle in a double well has many possible energy states. The qubit picture pretends only the lowest two matter. The code tests that by starting the particle in two different ways (**A** and **B**) and comparing the full simulation with the "only two states" prediction.

1. **A**: a mix of exactly the two lowest states (the initial state `psi_left` built in Section 5.4). The two-state prediction should be essentially perfect, so this just confirms the code works.
2. **B**: a Gaussian wave packet localised in the left well. This is not made of only two states, so the two-state picture should be slightly off. How far off gives an indication of how good the qubit approximation is.

**Variables (Section 6.2 Qubit check).**

| Name | Meaning |
|---|---|
| `dt_q`, `T_end_q` | time step and total time for this test |
| `run_PR` | function: full simulation, returns times, $P_R(t)$ and the total probability |
| `two_level_PR` | function: $P_R(t)$ predicted using only $\phi_0$ and $\phi_1$, and the population in those two states |
| `psi_start` | initial state passed to the functions |
| `c` | overlaps $\langle\phi_n\mid\psi\rangle$ of the initial state with $\phi_0$ and $\phi_1$ |
| `r` | matrix of overlaps of $\phi_n$ and $\phi_m$ over the right half ($x\gt 0$) |
| `PR` | running sum for the two-level $P_R(t)$ |
| `omega_well` | oscillation frequency at the bottom of one well, $\sqrt{V''(x_{\min})}$ |
| `sigma_well` | width of the harmonic-oscillator ground state in that well, $1/\sqrt{2\,\omega}$ |
| `psi_gauss` | Start B: normalised Gaussian placed at the left minimum |
| `qubit_results` | dictionary of `(times, full simulation, two-level)` for each start |
| `t_q`, `P_sim`, `P_two` | times, $P_R$ from the full simulation and $P_R$ from the two-level projection |
| `population` | probability contained in $\phi_0$ and $\phi_1$ |

In [ ]:
# Test the two-state (qubit) picture against the full simulation for two different starting states
dt_q = 0.05 # time step
T_end_q = 300.0# run time (covers the first transfer)

# runs full Crank-Nicolson simulation; returning times, P_right(t) and the total probability
def run_PR(psi_start):
    obs = lambda psi: (np.sum(np.abs(psi[x_dw > 0])**2) * dx_dw, total_probability(psi, dx_dw))
    r = crank_nicolson(psi_start, V_dw, dx_dw, dt_q, T_end_q, observe=obs, record_every=5)
    return r["times"], r["obs"][:, 0], r["obs"][:, 1]

# runs two state simulation, keeping only phi_0 and phi_1
def two_level_PR(psi_start, t):
    c = [np.sum(np.conjugate(phi[:, n]) * psi_start) * dx_dw for n in (0, 1)]      # overlaps <phi_n|psi>
    r = [[np.sum(np.conjugate(phi[x_dw > 0, n]) * phi[x_dw > 0, m]) * dx_dw for m in (0, 1)] for n in (0, 1)]
    PR = np.zeros_like(t, dtype=complex)
    for n in (0, 1):
        for m in (0, 1):
            PR += np.conjugate(c[n]) * c[m] * r[n][m] * np.exp(1j * (energies[n] - energies[m]) * t)
    return PR.real, sum(abs(ci)**2 for ci in c)

# A: the superposition of the two lowest states from Section 5.4 (exactly two-level)
# B: a Gaussian placed at the left minimum
omega_well = np.sqrt(12 * a * x_min**2 - 2 * b)# oscillation frequency at the bottom of one well
sigma_well = 1.0 / np.sqrt(2 * omega_well) # width of the harmonic-oscillator ground state
psi_gauss = np.exp(-(x_dw + x_min)**2 / (4 * sigma_well**2)).astype(complex)
psi_gauss /= np.sqrt(total_probability(psi_gauss, dx_dw))

# plotting both the superposition of first 2 states and the full Gaussian's prob. of being in right well over time
qubit_results = {}
for name, psi_start in [("Superposition (of first 2 states)", psi_left), ("Left-well Gaussian", psi_gauss)]:
    t_q, P_sim, _ = run_PR(psi_start)
    P_two, population = two_level_PR(psi_start, t_q)
    qubit_results[name] = (t_q, P_sim, P_two)
    print(f"{name:>34}: population in {{phi0, phi1}} = {population:.4f}, "
          f"max |simulation - two-level| = {np.max(np.abs(P_sim - P_two)):.4f}, "
          f"peak P_right: simulation {np.max(P_sim):.3f}, two-level {np.max(P_two):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, (name, (t_q, P_sim, P_two)) in zip(axes, qubit_results.items()):
    ax.plot(t_q, P_sim, label="full simulation")
    ax.plot(t_q, P_two, "r--", label="two-level projection")
    ax.set_xlabel("Time t"); ax.set_title(name); ax.legend()
axes[0].set_ylabel("Probability in right well")
plt.tight_layout(); plt.show()

#### Observations
- The plots show the probability of finding the particle in the right-hand well over time.
- **Left panel (Start A):** the solid and dashed lines are perfectly in line with each other (the largest difference is $0.0002$), so the two-state picture is perfect here. This is expected since this is what the notebook assumed that the first two states were a good model of the qubit system.
- **Right panel (Start B):** the dashed line is a smooth curve, while the solid line (simulation with all states) follows it roughly but with small fluctuations. The fluctuations are the particle leaking into higher states that the two-state picture ignores.
- About 95% of the probability is in the two lowest states, so the qubit picture is a good approximation. The other 5% causes the fluctuation and the small difference (up to about $0.06$).


**So the double well works as a qubit-like two-level system to a few per cent for this initial state, with the remainder set by how well the starting state matches $\phi_0$ and $\phi_1$.**

## 7. Discussion and conclusion
### 7.1 Discussion
**Rectangular barrier.** The simulation produces a non-zero transmission probability even though the wave packet's mean energy is below the barrier height, consistent with quantum tunnelling. The numerical transmission for the original barrier ($T=0.0794$) was not equal to the analytical single-energy value ($T=0.0707$). Two effects account for the difference:
- **Energy spread.** The Gaussian packet contains a range of energies (not just a single energy), and because $T(E)$ curves upwards, averaging over the packet raises the expected transmission to $T=0.0861$.
- **Barrier width.** The definition $|x|\le d/2$ placed one grid point too many in the barrier, making it $d+dx (2.05)$ wide instead of $d (2.0)$.


**Convergence of the barrier simulation.** Refining the grid affects the two barrier definitions differently:
- *Original barrier $2.05$:* its width $d+dx $ shrinks towards $d$ as $dx$ decreases, so $T$ keeps increasing ($0.0731\to0.0844$) and so $T$ has not converged. The error against the analytical value falls in proportion to $dx$ (first order, fitted slope $0.97$), because the width error is itself proportional to $dx$.
- *Edge-corrected barrier $2.00$:* the width is exactly $d $ on every grid, so only the ordinary discretisation error remains. It falls roughly in proportion to $dx^2$ (fitted slope $1.89$, close to second order) and is already only $1.8\times10^{-4}$ from the analytical value on the coarsest grid ($dx=0.1$).
- *Time step:* halving $dt$ from $0.05$ to $0.025$ changed $T$ by only about $10^{-11}$, so the time step does not limit the accuracy.
- So the grid spacing, not the time step, limits the accuracy in this barrier simulation.

**Numerical method.** The Crank–Nicolson method maintained the total probability of the wavefunction to a high degree of accuracy throughout both simulations (maximum deviations of about $10^{-12}$), supporting the stability and reliability of the numerical approach.

**Double well.** The two lowest eigenstates are close in energy ($\Delta E=0.0125$). A superposition of the two lowest eigenstates is initially localised in one well. Because the states have slightly different energies, their relative phase evolves, and the probability moves periodically between the wells. The right-well probability peaks at $0.997$ at $t\approx251.10$, in close agreement with the first transfer time $\pi/\Delta E=251.09$ estimated from the energy splitting. This demonstrates the link between stationary energy eigenstates and time-dependent tunnelling dynamics. The two-state (qubit) model reproduces the dynamics exactly for a superposition of the two lowest states, and to within about $0.06$ in $P_R$ for a Gaussian placed in one well (which has ~$95\%$ of its probability in those two states).

**Limitations.**
- The wavefunction is fixed at zero at $x=\pm L$, which acts like a hard wall. By $t=80$ about $0.1\%$ of the probability lies within 10 units of the walls, so their influence is small but not zero; longer runs would need a larger domain or absorbing boundaries.
- $R$ and $T$ are estimated at one final time, and the probability inside the barrier is excluded (hence why $R+T$ slightly below 1).
- Only one set of parameters was explored for each system.

### 7.2 Extending the code
- Replace the fixed boundaries (hard walls) with absorbing boundaries so that longer simulations are not affected by reflections from $x=\pm L$.
- Vary the packet's mean energy (changing `k0`), the barrier height `V0` and the width `d` (the function `barrier_transmission` already takes these as arguments) and compare the simulated $T$ with `T_averaged` to map out the transmission curve $T(E)$.
- Investigate a time-dependent drive of the double well, and phase-sensitive measures of qubit-like state transfer (the Crank–Nicolson matrix would then have to be rebuilt at each step).

### 7.3 Future Work
- **Numerical**: the first extension is a split-step Fourier integrator. Crank–Nicolson treats the kinetic and potential energy together. Split-step treats them separately, because the kinetic part is simple in momentum space and the potential part is simple in position space. A fast Fourier transform switches between the two, so no linear system has to be solved. I’d compare its accuracy and run time against Crank–Nicolson, and see which is better for this problem.
- **Quantum computing** : the second extension is quantum control. So far the particle moves between wells on its own, at a fixed timescale set by the energy splitting. In a real device you’d control this, for example by briefly tilting the potential with a pulse. I’d vary the pulse’s duration and amplitude to find where the particle transfers reliably, and where it ends up in an equal superposition, which is the starting point for most quantum operations.

## 8. References and GenAI statement
### References
1. Libretexts (2025) *Quantum Tunneling of Particles through Potential Barriers*. https://phys.libretexts.org/Bookshelves/University_Physics/University_Physics_(OpenStax)/University_Physics_III_Optics_and_Modern_Physics_(OpenStax)/07:_Quantum_Mechanics/7.07:_Quantum_Tunneling_of_Particles_through_Potential_Barriers
2. Bertlmann, A. Reinhold, and Nicolai Friis,(2023) 'The Time-Independent Schrödinger Equation', Modern Quantum Theory: From Quantum Mechanics to Entanglement and Quantum Information (Online edn, Oxford Academic), https://doi.org/10.1093/oso/9780199683338.003.0004.
3. J. Crank, and P. Nicolson,(1996) 'A practical method for numerical evaluation of solutions of partial differential equations of the heat-conduction type', *Adv Comput Math*, **6**, pp. 207–226. https://doi.org/10.1007/BF02127704 
4. SciPy documentation (User Guide): Linear Algebra (`scipy.linalg') https://docs.scipy.org/doc//scipy/tutorial/linalg.html 

### GenAI statement
> I used GenAI to help me understand the project brief and to give me feedback on my code. I also used it to help write me spcript to go alongside the presentation. I have attached a table (word document) with a record of all the prompts and links to the conversations I had with various GenAI.

## Appendix
### A. Animation of the wave packet before, during and after the barrier
An animation of the Gaussian wave packet before, during and after the collision with the barrier. It reuses the snapshots saved in Section 3.2 (one every 0.25 time units), so no simulation is repeated. The text in the top-left corner shows the time; the text in the top-right shows the probability to the left and right of the barrier.

**Variables (Appendix A).**

| Name | Meaning |
|---|---|
| `frames` | array of $\lvert \psi\rvert ^2$ for every saved snapshot (one row per animation frame) |
| `ymax_anim` | upper $y$-limit of the animation |
| `line` | the plotted line, whose $y$-data is updated in each frame |
| `time_text`, `prob_text` | text objects showing the time and the left/right probabilities |
| `update` | function that draws animation frame `k` |
| `k` | index of the current frame |
| `P_left_k`, `P_right_k` | probability to the left and right of the barrier in frame `k` |
| `anim` | the animation object |

In [ ]:
# Animate the wave packet meeting the barrier, using the snapshots saved in Section 3.2
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
plt.rcParams["animation.embed_limit"] = 100   # MB; allows a longer inline animation

frames = np.abs(snapshots)**2  # |psi|^2 of every saved snapshot (321 frames, 0.25 time units apart)
ymax_anim = 1.1 * frames.max()   # upper limit of the y-axis

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.fill_between(x, 0, (V / V0) * 0.55 * ymax_anim, color="C1", alpha=0.35, label="Barrier")
line, = ax.plot(x, frames[0], "C0-", lw=1.6)
time_text = ax.text(0.02, 0.90, "", transform=ax.transAxes)
prob_text = ax.text(0.98, 0.90, "", transform=ax.transAxes, ha="right")

ax.set_xlim(-60, 60)
ax.set_ylim(0, ymax_anim)
ax.set_xlabel("$x$")
ax.set_ylabel(r"$|\psi(x,t)|^2$")
ax.set_title("Wave packet meeting a rectangular potential barrier")
ax.legend(loc="upper center")

def update(k):
    """Update the line and the text for animation frame k."""
    line.set_ydata(frames[k])
    time_text.set_text(f"t = {snapshot_times[k]:.2f}")
    P_left_k = np.sum(frames[k][x < -d/2]) * dx
    P_right_k = np.sum(frames[k][x > d/2]) * dx
    prob_text.set_text(f"left: {P_left_k:.3f}   right: {P_right_k:.3f}")
    return line, time_text, prob_text

anim = FuncAnimation(fig, update, frames=len(frames), interval=40, blit=True)
plt.close(fig)  # stops a duplicate static figure appearing

HTML(anim.to_jshtml())  # plays inline in the notebook

### B. Wavenumber distribution of the Gaussian wave packet
Used in Section 4.2 to average the plane-wave transmission $T(E)$ over the energies in the packet.

The initial packet is
$$
\psi(x,0)\propto\exp\left[-\frac{(x-x_0)^2}{4\sigma^2}\right]e^{ik_0x}.
$$
Writing $\psi$ as a superposition of plane waves, $\psi(x,0)=\int\phi(k)\,e^{ikx}\,dk$, the amplitude $\phi(k)$ is the Fourier transform of $\psi(x,0)$. The Fourier transform of a Gaussian is another Gaussian, and the factor $e^{ik_0x}$ shifts its centre to $k_0$ (the factor $e^{-ikx_0}$ from the centre $x_0$ only changes the phase):
$$
|\phi(k)|\propto\exp\left[-\sigma^2(k-k_0)^2\right].
$$
Squaring gives the probability density of the packet in wavenumber:
$$
w(k)=|\phi(k)|^2\propto\exp\left[-2\sigma^2(k-k_0)^2\right]
=\exp\left[-\frac{(k-k_0)^2}{2\sigma_k^2}\right],
\qquad
\sigma_k=\frac{1}{2\sigma}.
$$
This is the weight used in `T_averaged`. For $\sigma=3$, $\sigma_k=1/6$.

Each wavenumber $k$ has energy $E_k=k^2/2$ and transmits with probability $T_{\mathrm{analytic}}(E_k)$, so the expected transmission of the packet is
$$
\langle T\rangle=\frac{\int T_{\mathrm{analytic}}(E_k)\,w(k)\,dk}{\int w(k)\,dk},
$$
where the denominator normalises the weights. The integral runs over $k>0$ only, since only right-moving components are incident on the barrier from the left. It is evaluated numerically with `trapezoid` over $k_0\pm8\sigma_k$. This gives $\langle T\rangle=0.0861$, compared with $T_{\mathrm{analytic}}(E=0.5)=0.0707$ for a single energy.